In [1]:
import sys
from pathlib import Path

# Add parent directory to path so 'src' imports work
sys.path.insert(0, str(Path.cwd().parent))

import json
import folium
from collections import Counter
from src.utils.file_utils import load_json

print("✓ Setup complete")

✓ Setup complete


In [2]:
cache_dir = Path("../data/raw/osm_cache")

categories = {}
for json_file in sorted(cache_dir.glob("hyd_*.json")):
    category = json_file.stem.replace("hyd_", "")
    data = load_json(json_file)
    categories[category] = data
    elem_count = len(data.get("elements", []))
    print(f"  {category:25s} → {elem_count:6d} elements")

  abandoned                 →     10 elements
  accident_junction         →    605 elements
  emergency_services        →   1547 elements
  industrial                →    255 elements
  poorly_lit_roads          →   4679 elements
  restricted                →    206 elements
  unsafe_transit            →     52 elements


In [3]:
print("\nElement type breakdown per category:\n")
print(f"{'Category':<25} {'node':>8} {'way':>8} {'relation':>10}")
print("-" * 55)

for cat_name, data in categories.items():
    type_counts = Counter(elem["type"] for elem in data.get("elements", []))
    nodes = type_counts.get("node", 0)
    ways = type_counts.get("way", 0)
    rels = type_counts.get("relation", 0)
    print(f"{cat_name:<25} {nodes:>8} {ways:>8} {rels:>10}")


Element type breakdown per category:

Category                      node      way   relation
-------------------------------------------------------
abandoned                        0       10          0
accident_junction              345      260          0
emergency_services            1386      161          0
industrial                       0      254          1
poorly_lit_roads                 0     4679          0
restricted                       0      204          2
unsafe_transit                  52        0          0


In [4]:
print("\nSample tags from each category:\n")

for cat_name, data in categories.items():
    elements = data.get("elements", [])
    if not elements:
        continue
    
    # Collect all tag keys used
    tag_keys = Counter()
    for elem in elements:
        for key in elem.get("tags", {}).keys():
            tag_keys[key] += 1
    
    print(f"\n[{cat_name.upper()}]")
    print(f"  Top 5 tag keys: {dict(tag_keys.most_common(5))}")
    
    # Show 1 sample element
    sample = elements[0]
    print(f"  Sample tags: {sample.get('tags', {})}")


Sample tags from each category:


[ABANDONED]
  Top 5 tag keys: {'landuse': 6, 'abandoned': 3, 'leisure': 3, 'building': 1}
  Sample tags: {'landuse': 'brownfield'}

[ACCIDENT_JUNCTION]
  Top 5 tag keys: {'highway': 605, 'junction': 260, 'name': 95, 'oneway': 94, 'crossing': 85}
  Sample tags: {'highway': 'traffic_signals', 'name': 'Neredmet Crossroad', 'source': 'AND', 'traffic_signals': 'signal'}

[EMERGENCY_SERVICES]
  Top 5 tag keys: {'amenity': 1547, 'name': 1471, 'addr:state': 1094, 'source': 1077, 'addr:district': 1035}
  Sample tags: {'amenity': 'police', 'name': 'Osmania University Police Station'}

[INDUSTRIAL]
  Top 5 tag keys: {'landuse': 219, 'name': 75, 'industrial': 39, 'man_made': 38, 'building': 26}
  Sample tags: {'landuse': 'industrial'}

[POORLY_LIT_ROADS]
  Top 5 tag keys: {'highway': 4679, 'surface': 4674, 'source': 464, 'access': 312, 'osm_type': 64}
  Sample tags: {'access': 'private', 'highway': 'residential', 'surface': 'unpaved'}

[RESTRICTED]
  Top 5 tag ke

In [5]:
# Create base map centered on Hyderabad
m = folium.Map(
    location=[17.385, 78.4867],
    zoom_start=11,
    tiles="OpenStreetMap"
)

# Color scheme per category
category_colors = {
    "industrial": "red",
    "abandoned": "purple",
    "poorly_lit_roads": "orange",
    "restricted": "darkred",
    "unsafe_transit": "blue",
    "accident_junction": "yellow",
    "emergency_services": "green",
}

print("Plotting on map...")
for cat_name, data in categories.items():
    color = category_colors.get(cat_name, "gray")
    plot_count = 0
    
    # Limit per category for performance
    for elem in data.get("elements", [])[:300]:
        try:
            if elem["type"] == "node":
                folium.CircleMarker(
                    location=[elem["lat"], elem["lon"]],
                    radius=3,
                    color=color,
                    fill=True,
                    fillOpacity=0.7,
                    popup=f"{cat_name}: {elem.get('tags', {}).get('name', 'N/A')}"
                ).add_to(m)
                plot_count += 1
            
            elif elem["type"] == "way" and "geometry" in elem:
                coords = [(g["lat"], g["lon"]) for g in elem["geometry"]]
                if len(coords) >= 3 and coords[0] == coords[-1]:
                    # Closed way → polygon
                    folium.Polygon(
                        locations=coords,
                        color=color,
                        fill=True,
                        fillOpacity=0.3,
                        popup=f"{cat_name}: {elem.get('tags', {}).get('name', 'N/A')}"
                    ).add_to(m)
                else:
                    # Open way → line
                    folium.PolyLine(
                        locations=coords,
                        color=color,
                        weight=2,
                        popup=f"{cat_name}"
                    ).add_to(m)
                plot_count += 1
        except Exception as e:
            pass
    
    print(f"  Plotted {plot_count} elements for {cat_name}")

# Display
m

Plotting on map...
  Plotted 10 elements for abandoned
  Plotted 300 elements for accident_junction
  Plotted 300 elements for emergency_services
  Plotted 254 elements for industrial
  Plotted 300 elements for poorly_lit_roads
  Plotted 204 elements for restricted
  Plotted 52 elements for unsafe_transit


In [6]:
output_path = Path("../data/intermediate/raw_data_visualization.html")
output_path.parent.mkdir(parents=True, exist_ok=True)
m.save(str(output_path))
print(f"✓ Map saved to: {output_path}")
print("Open this HTML file in a browser to view the full interactive map")

✓ Map saved to: ..\data\intermediate\raw_data_visualization.html
Open this HTML file in a browser to view the full interactive map
